In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [15]:
DATA_DIR = Path('C:/Users/223114186/Desktop/TP3_Word Embeddings/data')
train_df = pd.read_csv(DATA_DIR / 'height_weight_sex_training_set.csv')
test_df = pd.read_csv(DATA_DIR / 'height_weight_sex_test_set.csv')


In [16]:
train_df.head(), test_df.head()

(   Height  Weight     Sex
 0  165.65   35.41  Female
 1  148.53   74.45  Female
 2  167.04   81.22    Male
 3  161.54   71.47    Male
 4  174.31   78.18    Male,
        Height     Weight     Sex
 0  146.323241  59.861065  Female
 1  175.695412  77.863687    Male
 2  183.216164  72.131992    Male
 3  184.245269  77.546000    Male
 4  132.302261  55.188496  Female)

In [18]:
print("train_raw.shape =", train_df.shape)
print("test_raw.shape  =", test_df.shape)

train_raw.shape = (3000, 3)
test_raw.shape  = (205, 3)


In [20]:
train_df = (train_df
            .rename(columns=lambda c: c.strip())  # au cas où colonnes avec espaces
            .dropna(subset=["Height","Weight","Sex"])
            .drop_duplicates()
            .copy())
test_df  = (test_df
            .rename(columns=lambda c: c.strip())
            .dropna(subset=["Height","Weight","Sex"])
            .drop_duplicates()
            .copy())

print("After basic clean - train_df.shape:", train_df.shape)


After basic clean - train_df.shape: (3000, 3)


In [21]:
sex_norm = train_df["Sex"].astype(str).str.strip().str.lower()
to_male = {"male", "m", "man", "1", "true"}
to_fem  = {"female", "f", "woman", "0", "false"}

def encode_sex(s):
    if s in to_male: return 1
    if s in to_fem:  return 0
    return np.nan

train_df["label"] = sex_norm.apply(encode_sex)

before = len(train_df)
train_df = train_df.dropna(subset=["label"]).copy()
train_df["label"] = train_df["label"].astype(np.int32)
after = len(train_df)
print(f"Rows kept after label mapping: {after}/{before}")

Rows kept after label mapping: 3000/3000


In [22]:
X = train_df[["Height","Weight"]].to_numpy(dtype=np.float32)
y = train_df["label"].to_numpy(dtype=np.int32)


In [23]:
# Sanity checks
unique, counts = np.unique(y, return_counts=True)
print("Classes et effectifs :", dict(zip(unique, counts)))
assert len(X) == len(y) and len(X) > 0, (X.shape, y.shape)

# 4) Split TRAIN -> (train/val) uniquement
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42,
    stratify=y if len(unique) == 2 else None
)


Classes et effectifs : {np.int32(0): np.int64(1500), np.int32(1): np.int64(1500)}


In [24]:
scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train)
X_val   = scaler.transform(X_val)

In [25]:
X_test = scaler.transform(test_df[["Height","Weight"]].to_numpy(dtype=np.float32))


In [26]:
model = keras.Sequential([
layers.Input(shape=(2,)),
layers.Dense(16, activation='relu'),
layers.Dense(2, activation='softmax')
])
model.compile(optimizer=keras.optimizers.Adam(1e-3),
loss='sparse_categorical_crossentropy',
metrics=['accuracy'])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │            48 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │            34 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 82 (328.00 B)

 Trainable params: 82 (328.00 B)

 Non-trainable params: 0 (0.00 B)

In [27]:
history = model.fit(X_train, y_train, epochs=30, batch_size=32,
validation_data=(X_val, y_val))


val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
print(f"Validation accuracy: {val_acc:.3f}")


# Exemples manuels
examples = np.array([[170, 60], [185, 90]], dtype=np.float32)
examples = scaler.transform(examples)
proba = model.predict(examples)
print('Probas exemples:', proba)


# Prédire sur le test (si labels absents, on sort juste les proba)
proba_test = model.predict(X_test)
pred_test = proba_test.argmax(axis=1)

Epoch 1/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6309 - loss: 0.6776 - val_accuracy: 0.8750 - val_loss: 0.5672
Epoch 2/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8671 - loss: 0.4692 - val_accuracy: 0.8767 - val_loss: 0.5601
Epoch 3/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8694 - loss: 0.3996 - val_accuracy: 0.8733 - val_loss: 0.6077
Epoch 4/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8669 - loss: 0.3567 - val_accuracy: 0.8750 - val_loss: 0.6665
Epoch 5/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8812 - loss: 0.3248 - val_accuracy: 0.8750 - val_loss: 0.7149
Epoch 6/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8772 - loss: 0.3265 - val_accuracy: 0.8733 - val_loss: 0.7513
Epoch 7/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8638 - loss: 0.3272 - val_accuracy: 0.8717 - val_loss: 0.7775
Epoch 8/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8757 - loss: 0.3077 - val_accuracy: 0.8717 - val_loss:

In [34]:
# Créer le dossier s'il n'existe pas
Path("C:/Users/223114186/Desktop/TP3_Word Embeddings/artifact").mkdir(parents=True, exist_ok=True)
Path_artifact = Path("C:/Users/223114186/Desktop/TP3_Word Embeddings/artifact")
# Sauvegarder ensuite le modèle et le scaler
model.save(Path_artifact / 'baseline_sex_classifier.keras')
np.savez(Path_artifact / 'height_weight_scaler.npz',
         mean_=scaler.mean_,
         scale_=scaler.scale_)
print("✅ Modèle et scaler sauvegardés dans ../artifacts/")

✅ Modèle et scaler sauvegardés dans ../artifacts/
